# ECSC Developmental Analysis v3: Robust Long-Range Half-Life

**Core robustness features:**
1. Cumulative-min envelope for non-monotonic curves
2. Minimum benefit filter (benefit_eps)
3. Fixed max context (512 primary, 1024 secondary) for long-range structure
4. Sample counts per context length
5. Bootstrap confidence intervals

**Additional enhancements:**
6. Length-matched summaries (subset analyses, quintile bins, regression with n_tokens covariate)
7. CI bands on perplexity curves + N heatmap
8. Within-child longitudinal analysis (if repeated measures available)

In [ ]:
!pip install -q torch transformers accelerate bitsandbytes scipy

In [ ]:
import torch
import numpy as np
import pandas as pd
import json
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from scipy import stats
from tqdm.auto import tqdm
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

class Config:
    # Data
    MIN_WORDS = 200

    # Context ablation
    MIN_TARGET_SIZE = 20
    TARGET_FRACTION = 0.10

    # Robust half-life parameters
    MAX_CONTEXT_FIXED = 512   # Primary fixed max for cross-doc comparison (long-range)
    MAX_CONTEXT_LONG = 1024   # Secondary (for docs that support it)
    BENEFIT_EPS = 1.0         # Minimum total benefit (perplexity units)

    # Bootstrap
    N_BOOTSTRAP = 1000
    CI_LEVEL = 0.95

    # Plotting
    MIN_DOCS_FOR_PLOT = 10   # Minimum docs contributing to show a point

config = Config()
print(f"Fixed max context: {config.MAX_CONTEXT_FIXED} tokens (primary)")
print(f"Long max context: {config.MAX_CONTEXT_LONG} tokens (secondary, if available)")
print(f"Minimum benefit threshold: {config.BENEFIT_EPS} perplexity units")

In [ ]:
from google.colab import files

print("Upload transcripts.jsonl:")
uploaded = files.upload()
DATA_FILE = list(uploaded.keys())[0]
print(f"Uploaded: {DATA_FILE}")

In [ ]:
# Load data
records = []
with open(DATA_FILE) as f:
    for line in f:
        record = json.loads(line)
        record['pop'] = json.loads(record['population'])
        record['word_count'] = len(record['text'].split())
        record['age_months'] = record['pop']['age_months']
        records.append(record)

df_all = pd.DataFrame(records)
df = df_all[df_all['word_count'] >= config.MIN_WORDS].copy()

def age_bin(age):
    if age < 72: return '4-6yr'
    elif age < 96: return '6-8yr'
    elif age < 120: return '8-10yr'
    else: return '10+yr'

df['age_group'] = df['age_months'].apply(age_bin)

print(f"Documents: {len(df)} (of {len(df_all)} total)")
print(f"\nBy age group:")
for ag in ['4-6yr', '6-8yr', '8-10yr', '10+yr']:
    n = len(df[df['age_group'] == ag])
    if n > 0:
        print(f"  {ag}: n={n}")

In [ ]:
MODEL_NAME = "mistralai/Mistral-7B-v0.1"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map="auto"
)
model.eval()
print(f"Loaded {MODEL_NAME}")

In [ ]:
# ============================================================
# CORE FUNCTIONS
# ============================================================

@torch.no_grad()
def compute_perplexity_on_region(token_ids, target_start, target_end):
    """Compute perplexity on tokens in [target_start, target_end]."""
    if target_end > len(token_ids):
        target_end = len(token_ids)
    
    input_ids = torch.tensor([token_ids], device=model.device)
    outputs = model(input_ids)
    logits = outputs.logits[0]
    
    total_loss = 0.0
    count = 0
    
    for i in range(target_start, target_end - 1):
        log_probs = torch.log_softmax(logits[i], dim=-1)
        target_token = token_ids[i + 1]
        token_loss = -log_probs[target_token].item()
        total_loss += token_loss
        count += 1
    
    if count == 0:
        return float('inf'), 0
    
    return np.exp(total_loss / count), total_loss / count


def get_context_lengths(max_context):
    """Generate context lengths with dense sampling at short range."""
    lengths = []
    # Every 4 tokens up to 32
    lengths.extend(range(4, min(33, max_context + 1), 4))
    # Every 16 tokens from 48 to 128
    lengths.extend(range(48, min(129, max_context + 1), 16))
    # Every 32 tokens beyond
    lengths.extend(range(160, max_context + 1, 32))
    return sorted(set(lengths))


def analyze_document_adaptive(text):
    """Run context ablation on a document."""
    full_tokens = tokenizer.encode(text)
    n_tokens = len(full_tokens)
    
    # Adaptive target size
    target_size = max(config.MIN_TARGET_SIZE, int(n_tokens * config.TARGET_FRACTION))
    target_size = min(target_size, n_tokens - 8)
    
    if target_size < config.MIN_TARGET_SIZE:
        return [], {}
    
    max_context = n_tokens - target_size
    context_lengths = get_context_lengths(max_context)
    
    if len(context_lengths) < 3:
        return [], {}
    
    results = []
    for ctx_len in context_lengths:
        doc_start = n_tokens - target_size - ctx_len
        truncated_tokens = full_tokens[doc_start:]
        
        target_start = len(truncated_tokens) - target_size
        target_end = len(truncated_tokens)
        
        ppl, loss = compute_perplexity_on_region(truncated_tokens, target_start, target_end)
        
        results.append({
            'context_length': ctx_len,
            'perplexity': ppl,
            'loss': loss,
        })
    
    meta = {
        'n_tokens': n_tokens,
        'target_size': target_size,
        'max_context_available': max_context,
    }
    
    return results, meta

In [ ]:
# ============================================================
# ROBUST HALF-LIFE COMPUTATION
# ============================================================

def compute_cumulative_min(contexts, perplexities):
    """
    Convert to monotone non-increasing curve using cumulative minimum.
    ppl_mon[i] = min(ppl[0:i+1])
    """
    # Sort by context length
    order = np.argsort(contexts)
    contexts = np.array(contexts)[order]
    perplexities = np.array(perplexities)[order]
    
    # Cumulative minimum
    ppl_mon = np.minimum.accumulate(perplexities)
    
    return contexts, ppl_mon


def interpolate_perplexity(contexts, perplexities, target_ctx):
    """Linearly interpolate perplexity at a target context length."""
    if target_ctx <= contexts[0]:
        return perplexities[0]
    if target_ctx >= contexts[-1]:
        return perplexities[-1]
    
    # Find bracketing points
    for i in range(len(contexts) - 1):
        if contexts[i] <= target_ctx <= contexts[i+1]:
            frac = (target_ctx - contexts[i]) / (contexts[i+1] - contexts[i])
            return perplexities[i] + frac * (perplexities[i+1] - perplexities[i])
    
    return perplexities[-1]


def compute_half_life_robust(contexts, perplexities, max_ctx_fixed=None, benefit_eps=1.0):
    """
    Compute half-life with:
    1. Cumulative-min envelope (handles non-monotonic curves)
    2. Minimum benefit filter
    3. Optional fixed max context
    
    Returns dict with all metrics.
    """
    # Convert to monotonic curve
    contexts, ppl_mon = compute_cumulative_min(contexts, perplexities)
    
    # Perplexity at minimum context (4 tokens or first available)
    ppl_at_min = ppl_mon[0]
    ctx_min = contexts[0]
    
    # Determine max context to use
    max_ctx_available = contexts[-1]
    if max_ctx_fixed is not None:
        max_ctx_used = min(max_ctx_available, max_ctx_fixed)
    else:
        max_ctx_used = max_ctx_available
    
    # Interpolate perplexity at max_ctx_used
    ppl_at_max = interpolate_perplexity(contexts, ppl_mon, max_ctx_used)
    
    # Total benefit
    total_benefit = ppl_at_min - ppl_at_max
    
    # Check minimum benefit threshold
    benefit_ok = total_benefit >= benefit_eps
    
    # Compute half-life (only on monotonic curve, up to max_ctx_used)
    half_life = np.nan
    if benefit_ok:
        target_ppl = ppl_at_min - 0.5 * total_benefit
        
        # Find crossing on monotonic curve
        for i in range(len(ppl_mon) - 1):
            if contexts[i+1] > max_ctx_used:
                break
            if ppl_mon[i] >= target_ppl >= ppl_mon[i+1]:
                frac = (ppl_mon[i] - target_ppl) / (ppl_mon[i] - ppl_mon[i+1])
                half_life = contexts[i] + frac * (contexts[i+1] - contexts[i])
                break
    
    # Compute perplexity at 32 tokens (for early metrics)
    ppl_at_32 = interpolate_perplexity(contexts, ppl_mon, 32) if 32 <= max_ctx_available else np.nan
    
    # Early drop (4 -> 32)
    early_drop = ppl_at_min - ppl_at_32 if not np.isnan(ppl_at_32) else np.nan
    early_drop_pct = 100 * early_drop / total_benefit if (benefit_ok and not np.isnan(early_drop)) else np.nan
    
    # Early slope (linear fit on monotonic curve, 4-32 tokens)
    early_mask = (contexts >= ctx_min) & (contexts <= 32)
    if early_mask.sum() >= 3:
        slope, _, _, _, _ = stats.linregress(contexts[early_mask], ppl_mon[early_mask])
        early_slope = slope
    else:
        early_slope = np.nan
    
    return {
        'ppl_at_min_ctx': ppl_at_min,
        'ppl_at_32': ppl_at_32,
        'ppl_at_max_ctx': ppl_at_max,
        'total_benefit': total_benefit,
        'benefit_ok': benefit_ok,
        'half_life': half_life,
        'early_drop': early_drop,
        'early_drop_pct': early_drop_pct,
        'early_slope': early_slope,
        'max_ctx_available': max_ctx_available,
        'max_ctx_used': max_ctx_used,
        'ctx_min': ctx_min,
        # Store curves for later
        'contexts': contexts.tolist(),
        'ppl_mon': ppl_mon.tolist(),
    }

In [ ]:
# ============================================================
# RUN ANALYSIS
# ============================================================

all_results = []
doc_metrics = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing"):
    doc_results, meta = analyze_document_adaptive(row['text'])
    
    if not doc_results or len(doc_results) < 3:
        continue
    
    # Store raw results
    for r in doc_results:
        r['doc_id'] = row['doc_id']
        r['age_months'] = row['age_months']
        r['age_group'] = row['age_group']
        r['word_count'] = row['word_count']
        all_results.append(r)
    
    # Compute robust metrics
    contexts = [r['context_length'] for r in doc_results]
    perplexities = [r['perplexity'] for r in doc_results]
    
    # With primary fixed max context (512)
    metrics_fixed = compute_half_life_robust(
        contexts, perplexities, 
        max_ctx_fixed=config.MAX_CONTEXT_FIXED,
        benefit_eps=config.BENEFIT_EPS
    )
    
    # With long fixed max context (1024) - only if doc supports it
    max_ctx_available = meta['max_context_available']
    if max_ctx_available >= config.MAX_CONTEXT_LONG:
        metrics_long = compute_half_life_robust(
            contexts, perplexities,
            max_ctx_fixed=config.MAX_CONTEXT_LONG,
            benefit_eps=config.BENEFIT_EPS
        )
        supports_long = True
    else:
        metrics_long = {k: np.nan for k in metrics_fixed.keys()}
        metrics_long['benefit_ok'] = False
        supports_long = False
    
    # With document max context (for comparison)
    metrics_docmax = compute_half_life_robust(
        contexts, perplexities,
        max_ctx_fixed=None,
        benefit_eps=config.BENEFIT_EPS
    )
    
    doc_metrics.append({
        'doc_id': row['doc_id'],
        'age_months': row['age_months'],
        'age_group': row['age_group'],
        'word_count': row['word_count'],
        'n_tokens': meta['n_tokens'],
        'target_size': meta['target_size'],
        'max_ctx_available': max_ctx_available,
        
        # Primary fixed max context metrics (512)
        'ppl_min_ctx': metrics_fixed['ppl_at_min_ctx'],
        'ppl_32': metrics_fixed['ppl_at_32'],
        'ppl_max_fixed': metrics_fixed['ppl_at_max_ctx'],
        'total_benefit_fixed': metrics_fixed['total_benefit'],
        'benefit_ok_fixed': metrics_fixed['benefit_ok'],
        'half_life_fixed': metrics_fixed['half_life'],
        'early_drop': metrics_fixed['early_drop'],
        'early_drop_pct_fixed': metrics_fixed['early_drop_pct'],
        'early_slope': metrics_fixed['early_slope'],
        'max_ctx_used_fixed': metrics_fixed['max_ctx_used'],
        
        # Long fixed max context metrics (1024)
        'supports_long_ctx': supports_long,
        'ppl_max_long': metrics_long['ppl_at_max_ctx'] if supports_long else np.nan,
        'total_benefit_long': metrics_long['total_benefit'] if supports_long else np.nan,
        'benefit_ok_long': metrics_long['benefit_ok'] if supports_long else False,
        'half_life_long': metrics_long['half_life'] if supports_long else np.nan,
        'early_drop_pct_long': metrics_long['early_drop_pct'] if supports_long else np.nan,
        
        # Doc max context metrics (for within-doc analysis)
        'ppl_max_docmax': metrics_docmax['ppl_at_max_ctx'],
        'total_benefit_docmax': metrics_docmax['total_benefit'],
        'benefit_ok_docmax': metrics_docmax['benefit_ok'],
        'half_life_docmax': metrics_docmax['half_life'],
        'early_drop_pct_docmax': metrics_docmax['early_drop_pct'],
    })

results_df = pd.DataFrame(all_results)
metrics_df = pd.DataFrame(doc_metrics)

print(f"\nProcessed {len(metrics_df)} documents")
print(f"\nBenefit OK rates:")
print(f"  Fixed ({config.MAX_CONTEXT_FIXED}): {metrics_df['benefit_ok_fixed'].sum()} ({100*metrics_df['benefit_ok_fixed'].mean():.1f}%)")
print(f"  Long ({config.MAX_CONTEXT_LONG}): {metrics_df['benefit_ok_long'].sum()} ({100*metrics_df['benefit_ok_long'].mean():.1f}%) - of {metrics_df['supports_long_ctx'].sum()} docs that support it")
print(f"  Docmax: {metrics_df['benefit_ok_docmax'].sum()} ({100*metrics_df['benefit_ok_docmax'].mean():.1f}%)")

In [ ]:
# ============================================================
# SAMPLE COUNTS PER CONTEXT LENGTH
# ============================================================

print("Sample counts per context length by age group:")
print("(for identifying thin tails)\n")

age_groups = ['4-6yr', '6-8yr', '8-10yr', '10+yr']
context_counts = {}

for ag in age_groups:
    ag_df = results_df[results_df['age_group'] == ag]
    counts = ag_df.groupby('context_length')['doc_id'].nunique()
    context_counts[ag] = counts

# Display
all_contexts = sorted(results_df['context_length'].unique())
print(f"{'Context':<10}", end='')
for ag in age_groups:
    print(f"{ag:>10}", end='')
print()
print("-" * 50)

for ctx in all_contexts:
    print(f"{ctx:<10}", end='')
    for ag in age_groups:
        n = context_counts[ag].get(ctx, 0)
        flag = '*' if n < config.MIN_DOCS_FOR_PLOT else ''
        print(f"{n:>9}{flag}", end='')
    print()

print(f"\n* = fewer than {config.MIN_DOCS_FOR_PLOT} docs (will be faded in plots)")

In [ ]:
# ============================================================
# BOOTSTRAP CONFIDENCE INTERVALS
# ============================================================

def bootstrap_ci(values, n_bootstrap=1000, ci=0.95, statistic=np.mean):
    """Compute bootstrap CI for a statistic."""
    values = np.array(values)
    values = values[~np.isnan(values)]
    
    if len(values) < 3:
        return np.nan, np.nan, np.nan
    
    rng = np.random.default_rng(42)
    boot_stats = []
    
    for _ in range(n_bootstrap):
        sample = rng.choice(values, size=len(values), replace=True)
        boot_stats.append(statistic(sample))
    
    alpha = (1 - ci) / 2
    ci_low = np.percentile(boot_stats, alpha * 100)
    ci_high = np.percentile(boot_stats, (1 - alpha) * 100)
    
    return statistic(values), ci_low, ci_high


print("Computing bootstrap CIs for key metrics...")

bootstrap_results = []

for ag in age_groups:
    ag_df = metrics_df[metrics_df['age_group'] == ag]
    
    if len(ag_df) < 5:
        continue
    
    result = {'age_group': ag, 'n': len(ag_df)}
    
    # Primary fixed max (512) metrics
    for metric in ['half_life_fixed', 'early_slope', 'early_drop_pct_fixed', 'ppl_min_ctx']:
        mean, ci_low, ci_high = bootstrap_ci(
            ag_df[metric].values, 
            n_bootstrap=config.N_BOOTSTRAP,
            ci=config.CI_LEVEL
        )
        result[f'{metric}_mean'] = mean
        result[f'{metric}_ci_low'] = ci_low
        result[f'{metric}_ci_high'] = ci_high
    
    # Long fixed max (1024) metrics - only for docs that support it
    ag_long_df = ag_df[ag_df['supports_long_ctx']]
    result['n_long'] = len(ag_long_df)
    
    if len(ag_long_df) >= 5:
        for metric in ['half_life_long', 'early_drop_pct_long']:
            mean, ci_low, ci_high = bootstrap_ci(
                ag_long_df[metric].values,
                n_bootstrap=config.N_BOOTSTRAP,
                ci=config.CI_LEVEL
            )
            result[f'{metric}_mean'] = mean
            result[f'{metric}_ci_low'] = ci_low
            result[f'{metric}_ci_high'] = ci_high
    else:
        for metric in ['half_life_long', 'early_drop_pct_long']:
            result[f'{metric}_mean'] = np.nan
            result[f'{metric}_ci_low'] = np.nan
            result[f'{metric}_ci_high'] = np.nan
    
    bootstrap_results.append(result)

bootstrap_df = pd.DataFrame(bootstrap_results)
print("Done.")

In [ ]:
# ============================================================
# RESULTS SUMMARY
# ============================================================

print("=" * 70)
print(f"RESULTS SUMMARY (primary fixed max context = {config.MAX_CONTEXT_FIXED} tokens)")
print("=" * 70)

print("\n--- Half-Life (fixed max context) ---")
for _, row in bootstrap_df.iterrows():
    print(f"{row['age_group']}: {row['half_life_fixed_mean']:.1f} "
          f"[{row['half_life_fixed_ci_low']:.1f}, {row['half_life_fixed_ci_high']:.1f}] "
          f"(n={row['n']})")

print("\n--- Early Slope (4-32 tokens) ---")
for _, row in bootstrap_df.iterrows():
    print(f"{row['age_group']}: {row['early_slope_mean']:.3f} "
          f"[{row['early_slope_ci_low']:.3f}, {row['early_slope_ci_high']:.3f}]")

print("\n--- Early Drop % (4→32 as % of 4→fixed max) ---")
for _, row in bootstrap_df.iterrows():
    print(f"{row['age_group']}: {row['early_drop_pct_fixed_mean']:.1f}% "
          f"[{row['early_drop_pct_fixed_ci_low']:.1f}, {row['early_drop_pct_fixed_ci_high']:.1f}]")

print("\n--- Perplexity @ Minimal Context ---")
for _, row in bootstrap_df.iterrows():
    print(f"{row['age_group']}: {row['ppl_min_ctx_mean']:.1f} "
          f"[{row['ppl_min_ctx_ci_low']:.1f}, {row['ppl_min_ctx_ci_high']:.1f}]")

# Long context results (if available)
has_long = bootstrap_df['n_long'].sum() > 0
if has_long:
    print("\n" + "=" * 70)
    print(f"LONG CONTEXT RESULTS (fixed max = {config.MAX_CONTEXT_LONG} tokens)")
    print("=" * 70)
    
    print("\n--- Half-Life (long context) ---")
    for _, row in bootstrap_df.iterrows():
        if row['n_long'] >= 5 and not np.isnan(row['half_life_long_mean']):
            print(f"{row['age_group']}: {row['half_life_long_mean']:.1f} "
                  f"[{row['half_life_long_ci_low']:.1f}, {row['half_life_long_ci_high']:.1f}] "
                  f"(n={row['n_long']})")
        else:
            print(f"{row['age_group']}: insufficient docs (n={row['n_long']})")
    
    print("\n--- Early Drop % (4→32 as % of 4→1024) ---")
    for _, row in bootstrap_df.iterrows():
        if row['n_long'] >= 5 and not np.isnan(row['early_drop_pct_long_mean']):
            print(f"{row['age_group']}: {row['early_drop_pct_long_mean']:.1f}% "
                  f"[{row['early_drop_pct_long_ci_low']:.1f}, {row['early_drop_pct_long_ci_high']:.1f}]")
        else:
            print(f"{row['age_group']}: insufficient docs")

In [ ]:
# ============================================================
# STATISTICAL TESTS
# ============================================================

print("\n" + "=" * 70)
print("STATISTICAL TESTS")
print("=" * 70)

# Only use docs with valid benefit
valid_df = metrics_df[metrics_df['benefit_ok_fixed']].copy()
print(f"\nUsing {len(valid_df)} docs with sufficient benefit (>= {config.BENEFIT_EPS} ppl units)")

print("\n--- Correlations with Age (Spearman) ---")
for metric in ['half_life_fixed', 'early_slope', 'early_drop_pct_fixed', 'ppl_min_ctx']:
    valid = valid_df[[metric, 'age_months']].dropna()
    if len(valid) > 10:
        rho, p = stats.spearmanr(valid['age_months'], valid[metric])
        sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
        print(f"{metric:<25}: rho={rho:+.3f}, p={p:.4f} {sig}")

print("\n--- Kruskal-Wallis (overall age group effect) ---")
for metric in ['half_life_fixed', 'early_slope', 'early_drop_pct_fixed', 'ppl_min_ctx']:
    groups = [valid_df[valid_df['age_group'] == ag][metric].dropna().values 
              for ag in age_groups if len(valid_df[valid_df['age_group'] == ag]) > 0]
    groups = [g for g in groups if len(g) >= 3]
    
    if len(groups) >= 2:
        h, p = stats.kruskal(*groups)
        sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
        print(f"{metric:<25}: H={h:.2f}, p={p:.4f} {sig}")

In [ ]:
# ============================================================
# ENHANCEMENT 1: LENGTH-MATCHED SUMMARIES
# ============================================================

print("=" * 70)
print("LENGTH-MATCHED ANALYSIS")
print("=" * 70)

# --- Subset analyses by context availability ---
print("\n--- Results by Context Availability Subset ---")

subsets = {
    'All docs': metrics_df,
    f'ctx >= {config.MAX_CONTEXT_FIXED}': metrics_df[metrics_df['max_ctx_available'] >= config.MAX_CONTEXT_FIXED],
    f'ctx >= {config.MAX_CONTEXT_LONG}': metrics_df[metrics_df['max_ctx_available'] >= config.MAX_CONTEXT_LONG],
}

for subset_name, subset_df in subsets.items():
    print(f"\n{subset_name} (N={len(subset_df)}):")
    print(f"  {'Age Group':<10} {'N':>6} {'HL (fixed)':>12} {'Early Drop %':>14} {'PPL @ min':>12}")
    print("  " + "-" * 58)
    
    for ag in age_groups:
        ag_df = subset_df[subset_df['age_group'] == ag]
        if len(ag_df) >= 3:
            hl = ag_df['half_life_fixed'].dropna().mean()
            edp = ag_df['early_drop_pct_fixed'].dropna().mean()
            ppl = ag_df['ppl_min_ctx'].dropna().mean()
            print(f"  {ag:<10} {len(ag_df):>6} {hl:>12.1f} {edp:>14.1f} {ppl:>12.1f}")
        else:
            print(f"  {ag:<10} {len(ag_df):>6} {'N/A':>12} {'N/A':>14} {'N/A':>12}")

# --- Length-matched comparison using n_tokens bins ---
print("\n\n--- Length-Matched Comparison (n_tokens quintile bins) ---")

# Create quintile bins based on n_tokens
metrics_df['n_tokens_bin'] = pd.qcut(metrics_df['n_tokens'], q=5, labels=['Q1', 'Q2', 'Q3', 'Q4', 'Q5'])

# For each bin, compute age group means
print("\nHalf-Life (fixed) by Age Group and Token Quintile:")
pivot_hl = metrics_df.pivot_table(
    values='half_life_fixed', 
    index='n_tokens_bin', 
    columns='age_group', 
    aggfunc='mean'
)[age_groups]
print(pivot_hl.round(1).to_string())

# Length-matched means (average across bins)
print("\n\nLength-Matched Means (averaged across token quintiles):")
length_matched = pivot_hl.mean(axis=0)
print(length_matched.round(1).to_string())

# --- Regression with n_tokens as covariate ---
print("\n\n--- Regression Analysis (controlling for document length) ---")

from scipy.stats import pearsonr

# Prepare data
valid_reg = metrics_df[['age_months', 'n_tokens', 'half_life_fixed', 'early_drop_pct_fixed', 'ppl_min_ctx']].dropna()

# Simple OLS with n_tokens as covariate
# Using manual calculation to avoid sklearn dependency
X = valid_reg[['age_months', 'n_tokens']].values
X = np.column_stack([np.ones(len(X)), X])  # Add intercept

for metric in ['half_life_fixed', 'early_drop_pct_fixed', 'ppl_min_ctx']:
    y = valid_reg[metric].values
    
    # OLS: beta = (X'X)^-1 X'y
    try:
        beta = np.linalg.lstsq(X, y, rcond=None)[0]
        
        # Compute residuals and standard errors
        y_pred = X @ beta
        residuals = y - y_pred
        n, p = X.shape
        mse = np.sum(residuals**2) / (n - p)
        var_beta = mse * np.linalg.inv(X.T @ X)
        se_beta = np.sqrt(np.diag(var_beta))
        
        # t-statistics
        t_stats = beta / se_beta
        
        # Partial correlation (age effect controlling for n_tokens)
        # Residualize both age and metric on n_tokens
        X_ntok = np.column_stack([np.ones(len(valid_reg)), valid_reg['n_tokens'].values])
        beta_age = np.linalg.lstsq(X_ntok, valid_reg['age_months'].values, rcond=None)[0]
        age_resid = valid_reg['age_months'].values - X_ntok @ beta_age
        
        beta_metric = np.linalg.lstsq(X_ntok, y, rcond=None)[0]
        metric_resid = y - X_ntok @ beta_metric
        
        partial_r, partial_p = pearsonr(age_resid, metric_resid)
        
        sig = "***" if partial_p < 0.001 else "**" if partial_p < 0.01 else "*" if partial_p < 0.05 else ""
        print(f"{metric:<25}: age coef={beta[1]:+.4f} (t={t_stats[1]:.2f}), "
              f"partial r={partial_r:+.3f}, p={partial_p:.4f} {sig}")
        print(f"{'':25}  n_tokens coef={beta[2]:+.4f} (t={t_stats[2]:.2f})")
    except Exception as e:
        print(f"{metric}: regression failed - {e}")

print("\nNote: Partial r shows age-metric correlation after controlling for document length.")

In [ ]:
# ============================================================
# ENHANCEMENT 2: VISUALIZATION WITH CI BANDS AND N ANNOTATIONS
# ============================================================

colors = {'4-6yr': '#e74c3c', '6-8yr': '#f39c12', '8-10yr': '#27ae60', '10+yr': '#3498db'}

# Compute bootstrap CI for perplexity curves
print("Computing bootstrap CIs for perplexity curves...")

def bootstrap_curve_ci(df, context_col, value_col, n_bootstrap=500, ci=0.95):
    """Compute bootstrap CI for mean at each context length."""
    contexts = sorted(df[context_col].unique())
    results = []
    
    rng = np.random.default_rng(42)
    
    for ctx in contexts:
        ctx_values = df[df[context_col] == ctx][value_col].values
        n_docs = len(ctx_values)
        
        if n_docs < 3:
            results.append({'context': ctx, 'mean': np.nan, 'ci_low': np.nan, 'ci_high': np.nan, 'n': n_docs})
            continue
        
        boot_means = []
        for _ in range(n_bootstrap):
            sample = rng.choice(ctx_values, size=len(ctx_values), replace=True)
            boot_means.append(np.mean(sample))
        
        alpha = (1 - ci) / 2
        results.append({
            'context': ctx,
            'mean': np.mean(ctx_values),
            'ci_low': np.percentile(boot_means, alpha * 100),
            'ci_high': np.percentile(boot_means, (1 - alpha) * 100),
            'n': n_docs
        })
    
    return pd.DataFrame(results)

# Compute CI for each age group
curve_cis = {}
for ag in age_groups:
    ag_df = results_df[results_df['age_group'] == ag]
    if len(ag_df) > 0:
        curve_cis[ag] = bootstrap_curve_ci(ag_df, 'context_length', 'perplexity', n_bootstrap=500)

print("Done.")

# Create figure
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# 1. Perplexity curves with CI bands and N annotations
ax = axes[0, 0]
for ag in age_groups:
    if ag not in curve_cis:
        continue
    
    ci_df = curve_cis[ag]
    color = colors.get(ag, 'gray')
    
    # Separate solid and faded regions
    solid_mask = ci_df['n'] >= config.MIN_DOCS_FOR_PLOT
    
    # Plot solid region with CI band
    solid_df = ci_df[solid_mask]
    if len(solid_df) > 0:
        ax.fill_between(solid_df['context'], solid_df['ci_low'], solid_df['ci_high'],
                        color=color, alpha=0.15)
        ax.plot(solid_df['context'], solid_df['mean'], 
                marker='o', color=color, label=ag, linewidth=2, markersize=4)
    
    # Plot faded region (thin sample)
    faded_df = ci_df[~solid_mask]
    if len(faded_df) > 0:
        ax.plot(faded_df['context'], faded_df['mean'],
                marker='o', color=color, linewidth=1, markersize=3, alpha=0.3)

ax.axvline(config.MAX_CONTEXT_FIXED, color='black', linestyle='--', alpha=0.5, label=f'Fixed max ({config.MAX_CONTEXT_FIXED})')
ax.axvline(config.MAX_CONTEXT_LONG, color='gray', linestyle=':', alpha=0.4, label=f'Long max ({config.MAX_CONTEXT_LONG})')
ax.set_xlabel('Context Length (tokens)')
ax.set_ylabel('Perplexity')
ax.set_title('Perplexity Curves with 95% CI\n(faded = N < 10)')
ax.legend(loc='upper right', fontsize=8)
ax.grid(True, alpha=0.3)

# 2. Half-life with CI (fixed max)
ax = axes[0, 1]
x_pos = range(len(bootstrap_df))
means = bootstrap_df['half_life_fixed_mean'].values
ci_low = bootstrap_df['half_life_fixed_ci_low'].values
ci_high = bootstrap_df['half_life_fixed_ci_high'].values
errors = [means - ci_low, ci_high - means]

bars = ax.bar(x_pos, means, yerr=errors, capsize=8,
              color=[colors.get(ag, 'gray') for ag in bootstrap_df['age_group']],
              alpha=0.7, edgecolor='black')
ax.set_xticks(x_pos)
ax.set_xticklabels([f"{row['age_group']}\n(n={row['n']})" for _, row in bootstrap_df.iterrows()])
ax.set_ylabel('Half-Life (tokens)')
ax.set_title(f'Half-Life with 95% CI\n(fixed max = {config.MAX_CONTEXT_FIXED})')
ax.grid(True, alpha=0.3, axis='y')

# 3. Early slope with CI
ax = axes[0, 2]
means = bootstrap_df['early_slope_mean'].values
ci_low = bootstrap_df['early_slope_ci_low'].values
ci_high = bootstrap_df['early_slope_ci_high'].values
errors = [means - ci_low, ci_high - means]

bars = ax.bar(x_pos, means, yerr=errors, capsize=8,
              color=[colors.get(ag, 'gray') for ag in bootstrap_df['age_group']],
              alpha=0.7, edgecolor='black')
ax.set_xticks(x_pos)
ax.set_xticklabels([f"{row['age_group']}" for _, row in bootstrap_df.iterrows()])
ax.set_ylabel('Slope (ppl/token)')
ax.set_title('Early Slope (4-32 tokens) with 95% CI')
ax.axhline(0, color='black', linestyle='-', alpha=0.3)
ax.grid(True, alpha=0.3, axis='y')

# 4. Early drop % with CI
ax = axes[1, 0]
means = bootstrap_df['early_drop_pct_fixed_mean'].values
ci_low = bootstrap_df['early_drop_pct_fixed_ci_low'].values
ci_high = bootstrap_df['early_drop_pct_fixed_ci_high'].values
errors = [means - ci_low, ci_high - means]

bars = ax.bar(x_pos, means, yerr=errors, capsize=8,
              color=[colors.get(ag, 'gray') for ag in bootstrap_df['age_group']],
              alpha=0.7, edgecolor='black')
ax.set_xticks(x_pos)
ax.set_xticklabels([f"{row['age_group']}" for _, row in bootstrap_df.iterrows()])
ax.set_ylabel('% of Total Benefit')
ax.set_title(f'Early Drop % (4→32 / 4→{config.MAX_CONTEXT_FIXED})')
ax.grid(True, alpha=0.3, axis='y')

# 5. Scatter: age vs half-life (fixed)
ax = axes[1, 1]
valid = metrics_df[metrics_df['benefit_ok_fixed']]
for ag in age_groups:
    ag_data = valid[valid['age_group'] == ag]
    ax.scatter(ag_data['age_months'], ag_data['half_life_fixed'], 
               alpha=0.5, label=ag, color=colors.get(ag, 'gray'), s=30)

# Trend line
z = np.polyfit(valid['age_months'], valid['half_life_fixed'], 1)
p = np.poly1d(z)
x_line = np.linspace(valid['age_months'].min(), valid['age_months'].max(), 100)
ax.plot(x_line, p(x_line), 'k--', alpha=0.7, linewidth=2)

ax.set_xlabel('Age (months)')
ax.set_ylabel('Half-Life (tokens)')
ax.set_title('Age vs Half-Life (fixed max)')
ax.legend()
ax.grid(True, alpha=0.3)

# 6. N per context length heatmap
ax = axes[1, 2]
# Create a simple visual of N dropping off
ctx_to_plot = [c for c in all_contexts if c <= config.MAX_CONTEXT_LONG]
n_matrix = np.zeros((len(age_groups), len(ctx_to_plot)))
for i, ag in enumerate(age_groups):
    for j, ctx in enumerate(ctx_to_plot):
        n_matrix[i, j] = context_counts[ag].get(ctx, 0)

im = ax.imshow(n_matrix, aspect='auto', cmap='YlOrRd')
ax.set_yticks(range(len(age_groups)))
ax.set_yticklabels(age_groups)
ax.set_xticks(range(0, len(ctx_to_plot), max(1, len(ctx_to_plot)//8)))
ax.set_xticklabels([ctx_to_plot[i] for i in range(0, len(ctx_to_plot), max(1, len(ctx_to_plot)//8))])
ax.set_xlabel('Context Length')
ax.set_ylabel('Age Group')
ax.set_title('N Docs per Context Length')
plt.colorbar(im, ax=ax, label='N docs')

plt.tight_layout()
plt.savefig('ecsc_age_analysis_v3_robust.png', dpi=150, bbox_inches='tight')
plt.show()

# Print N summary
print("\n--- Sample Size Summary ---")
print(f"Context lengths where ALL age groups have N >= {config.MIN_DOCS_FOR_PLOT}:")
safe_contexts = [ctx for ctx in all_contexts 
                 if all(context_counts[ag].get(ctx, 0) >= config.MIN_DOCS_FOR_PLOT for ag in age_groups)]
print(f"  Range: {min(safe_contexts) if safe_contexts else 'none'} - {max(safe_contexts) if safe_contexts else 'none'} tokens")

In [ ]:
# ============================================================
# COMPARISON: FIXED vs LONG vs DOCMAX HALF-LIFE
# ============================================================

print("\n" + "=" * 70)
print("COMPARISON: Fixed Max vs Long Max vs Document Max Half-Life")
print("=" * 70)

print(f"\nPrimary fixed max context: {config.MAX_CONTEXT_FIXED} tokens")
print(f"Long fixed max context: {config.MAX_CONTEXT_LONG} tokens")

print(f"\n{'Age Group':<12} {'HL (512)':>10} {'HL (1024)':>11} {'HL (docmax)':>12} {'n (1024)':>10}")
print("-" * 60)

for ag in age_groups:
    ag_df = metrics_df[metrics_df['age_group'] == ag]
    if len(ag_df) == 0:
        continue
    
    hl_fixed = ag_df['half_life_fixed'].dropna().mean()
    hl_docmax = ag_df['half_life_docmax'].dropna().mean()
    
    # Long context (only for docs that support it)
    ag_long_df = ag_df[ag_df['supports_long_ctx']]
    if len(ag_long_df) >= 5:
        hl_long = ag_long_df['half_life_long'].dropna().mean()
        n_long = len(ag_long_df)
        print(f"{ag:<12} {hl_fixed:>10.1f} {hl_long:>11.1f} {hl_docmax:>12.1f} {n_long:>10}")
    else:
        n_long = len(ag_long_df)
        print(f"{ag:<12} {hl_fixed:>10.1f} {'N/A':>11} {hl_docmax:>12.1f} {n_long:>10}")

print("\nNote: HL (1024) only computed for documents with sufficient context available.")
print("      Longer half-life indicates benefit spread over more distant context.")

# Early drop comparison
print("\n--- Early Drop % Comparison ---")
print(f"{'Age Group':<12} {'% (4→32/512)':>14} {'% (4→32/1024)':>14}")
print("-" * 45)

for ag in age_groups:
    ag_df = metrics_df[metrics_df['age_group'] == ag]
    if len(ag_df) == 0:
        continue
    
    edp_fixed = ag_df['early_drop_pct_fixed'].dropna().mean()
    
    ag_long_df = ag_df[ag_df['supports_long_ctx']]
    if len(ag_long_df) >= 5:
        edp_long = ag_long_df['early_drop_pct_long'].dropna().mean()
        print(f"{ag:<12} {edp_fixed:>14.1f} {edp_long:>14.1f}")
    else:
        print(f"{ag:<12} {edp_fixed:>14.1f} {'N/A':>14}")

print("\nNote: Lower % indicates benefit distributed over longer range (more long-range structure).")

In [ ]:
# ============================================================
# ENHANCEMENT 3: WITHIN-CHILD LONGITUDINAL ANALYSIS
# ============================================================

print("=" * 70)
print("WITHIN-CHILD LONGITUDINAL ANALYSIS")
print("=" * 70)

# Check if we have author_id to track children across years
# Merge metrics back with original df to get author_id and study_year
metrics_with_author = metrics_df.merge(
    df[['doc_id', 'author_id', 'pop']].assign(
        study_year=df['pop'].apply(lambda x: x.get('study_year', None) if isinstance(x, dict) else None)
    ),
    on='doc_id',
    how='left'
)

# Check for repeated measures
author_counts = metrics_with_author.groupby('author_id').size()
authors_with_multiple = author_counts[author_counts > 1]

print(f"\nTotal unique children (author_id): {metrics_with_author['author_id'].nunique()}")
print(f"Children with multiple observations: {len(authors_with_multiple)}")

if len(authors_with_multiple) >= 5:
    print(f"\n--- Paired Analysis: Within-Child Changes ---")
    
    # Get children with exactly 2 or more observations
    repeated_authors = authors_with_multiple.index.tolist()
    repeated_df = metrics_with_author[metrics_with_author['author_id'].isin(repeated_authors)].copy()
    
    # Sort by author and age to get trajectories
    repeated_df = repeated_df.sort_values(['author_id', 'age_months'])
    
    # Compute within-child changes
    paired_results = []
    
    for author_id in repeated_authors:
        child_df = repeated_df[repeated_df['author_id'] == author_id].sort_values('age_months')
        
        if len(child_df) < 2:
            continue
        
        # Take first and last observation for this child
        first = child_df.iloc[0]
        last = child_df.iloc[-1]
        
        age_diff = last['age_months'] - first['age_months']
        
        # Only include if there's meaningful age difference (at least 6 months)
        if age_diff >= 6:
            paired_results.append({
                'author_id': author_id,
                'age_first': first['age_months'],
                'age_last': last['age_months'],
                'age_diff': age_diff,
                'hl_first': first['half_life_fixed'],
                'hl_last': last['half_life_fixed'],
                'hl_change': last['half_life_fixed'] - first['half_life_fixed'],
                'edp_first': first['early_drop_pct_fixed'],
                'edp_last': last['early_drop_pct_fixed'],
                'edp_change': last['early_drop_pct_fixed'] - first['early_drop_pct_fixed'],
                'ppl_first': first['ppl_min_ctx'],
                'ppl_last': last['ppl_min_ctx'],
                'ppl_change': last['ppl_min_ctx'] - first['ppl_min_ctx'],
            })
    
    if paired_results:
        paired_df = pd.DataFrame(paired_results)
        
        print(f"\nChildren with ≥6 month longitudinal data: {len(paired_df)}")
        print(f"Mean age span: {paired_df['age_diff'].mean():.1f} months (range: {paired_df['age_diff'].min():.0f}-{paired_df['age_diff'].max():.0f})")
        
        print("\n--- Within-Child Change Statistics ---")
        
        for metric, change_col, expected_dir in [
            ('Half-Life', 'hl_change', 'increase'),
            ('Early Drop %', 'edp_change', 'decrease'),
            ('PPL @ min ctx', 'ppl_change', 'decrease'),
        ]:
            changes = paired_df[change_col].dropna()
            
            if len(changes) >= 5:
                mean_change = changes.mean()
                n_increase = (changes > 0).sum()
                n_decrease = (changes < 0).sum()
                
                # Wilcoxon signed-rank test (paired)
                try:
                    stat, p = stats.wilcoxon(changes)
                    sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
                except Exception:
                    stat, p = np.nan, np.nan
                    sig = ""
                
                # One-sample t-test
                t_stat, t_p = stats.ttest_1samp(changes, 0)
                
                direction = "↑" if mean_change > 0 else "↓" if mean_change < 0 else "→"
                expected = "✓" if (expected_dir == 'increase' and mean_change > 0) or \
                                  (expected_dir == 'decrease' and mean_change < 0) else "✗"
                
                print(f"\n{metric}:")
                print(f"  Mean change: {mean_change:+.2f} {direction} (expected: {expected_dir} {expected})")
                print(f"  Increased: {n_increase}/{len(changes)}, Decreased: {n_decrease}/{len(changes)}")
                print(f"  Wilcoxon signed-rank: W={stat:.1f}, p={p:.4f} {sig}")
                print(f"  One-sample t-test: t={t_stat:.2f}, p={t_p:.4f}")
        
        # Correlation between age span and change magnitude
        print("\n--- Change vs Age Span ---")
        for metric, change_col in [('Half-Life', 'hl_change'), ('Early Drop %', 'edp_change')]:
            valid = paired_df[['age_diff', change_col]].dropna()
            if len(valid) >= 5:
                r, p = stats.spearmanr(valid['age_diff'], valid[change_col])
                sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
                print(f"{metric} change vs age span: r={r:+.3f}, p={p:.4f} {sig}")
        
        # Save paired results
        paired_df.to_csv('ecsc_age_v3_longitudinal.csv', index=False)
        print("\nSaved: ecsc_age_v3_longitudinal.csv")
        
    else:
        print("\nNo children with ≥6 month age span between observations.")
        print("Longitudinal analysis not possible with current data.")

else:
    print("\nInsufficient repeated measures for longitudinal analysis.")
    print("This may be a cross-sectional dataset with one observation per child.")
    
    # Alternative: Check if study_year provides any longitudinal structure
    if 'study_year' in metrics_with_author.columns:
        year_counts = metrics_with_author.groupby('study_year').size()
        print(f"\nObservations by study_year: {dict(year_counts)}")
        print("Note: study_year indicates recruitment cohort, not repeated measures.")

In [ ]:
# ============================================================
# SAVE RESULTS
# ============================================================

import os

# Save metrics
metrics_df.to_csv('ecsc_age_v3_metrics.csv', index=False)

# Save bootstrap results
bootstrap_df.to_csv('ecsc_age_v3_bootstrap.csv', index=False)

# Save context counts
context_counts_df = pd.DataFrame(context_counts).fillna(0).astype(int)
context_counts_df.index.name = 'context_length'
context_counts_df.to_csv('ecsc_age_v3_context_counts.csv')

# Save curve CIs
curve_ci_all = pd.concat([
    ci_df.assign(age_group=ag) for ag, ci_df in curve_cis.items()
], ignore_index=True)
curve_ci_all.to_csv('ecsc_age_v3_curve_cis.csv', index=False)

# Save config
config_dict = {
    'MIN_WORDS': config.MIN_WORDS,
    'MAX_CONTEXT_FIXED': config.MAX_CONTEXT_FIXED,
    'MAX_CONTEXT_LONG': config.MAX_CONTEXT_LONG,
    'BENEFIT_EPS': config.BENEFIT_EPS,
    'N_BOOTSTRAP': config.N_BOOTSTRAP,
    'CI_LEVEL': config.CI_LEVEL,
}
with open('ecsc_age_v3_config.json', 'w') as f:
    json.dump(config_dict, f, indent=2)

print("Saved:")
print("  - ecsc_age_v3_metrics.csv")
print("  - ecsc_age_v3_bootstrap.csv")
print("  - ecsc_age_v3_context_counts.csv")
print("  - ecsc_age_v3_curve_cis.csv")
print("  - ecsc_age_v3_config.json")
print("  - ecsc_age_analysis_v3_robust.png")

# Check if longitudinal file was created
if os.path.exists('ecsc_age_v3_longitudinal.csv'):
    print("  - ecsc_age_v3_longitudinal.csv")

print(f"\nConfig: MAX_CONTEXT_FIXED={config.MAX_CONTEXT_FIXED}, MAX_CONTEXT_LONG={config.MAX_CONTEXT_LONG}")

In [ ]:
from google.colab import files
import os

files.download('ecsc_age_v3_metrics.csv')
files.download('ecsc_age_v3_bootstrap.csv')
files.download('ecsc_age_v3_context_counts.csv')
files.download('ecsc_age_v3_curve_cis.csv')
files.download('ecsc_age_v3_config.json')
files.download('ecsc_age_analysis_v3_robust.png')

# Download longitudinal file if it exists
if os.path.exists('ecsc_age_v3_longitudinal.csv'):
    files.download('ecsc_age_v3_longitudinal.csv')

## Enhancements Implemented

### Core Robustness (v3 base)
1. **Cumulative-min envelope**: `ppl_mon[i] = min(ppl[0:i+1])` prevents early-noise artifacts
2. **Minimum benefit filter**: Documents with `total_benefit < 1.0` ppl units flagged as `benefit_ok=False`
3. **Fixed max context for long-range structure**: 
   - Primary: `MAX_CONTEXT_FIXED=512` tokens - probes genuine long-range coherence
   - Secondary: `MAX_CONTEXT_LONG=1024` tokens - for docs that support it
4. **Sample counts per context length**: Thin tails identified and faded in plots
5. **Bootstrap CIs**: 1000 resamples, 95% CI for all key metrics

### Enhancement 1: Length-Matched Summaries
- **Subset analyses**: Results stratified by context availability (all docs, ≥512, ≥1024)
- **n_tokens quintile bins**: Age effects within each length bin to control confounds
- **Regression with covariate**: Partial correlations controlling for document length

### Enhancement 2: CI Bands on Curves
- **Bootstrap CI bands**: 95% confidence bands on perplexity curves
- **N annotations**: Sample size heatmap showing where data gets thin
- **Safe context range**: Identifies context lengths with adequate N across all age groups

### Enhancement 3: Within-Child Longitudinal Analysis
- **Repeated measures detection**: Identifies children with multiple observations
- **Paired analysis**: Computes within-child Δ half-life, Δ early-drop across time
- **Wilcoxon signed-rank test**: Tests if changes are significantly different from zero
- **Change vs age span correlation**: Tests if larger age gaps produce larger changes

## Output Files
- `ecsc_age_v3_metrics.csv` - Per-document metrics
- `ecsc_age_v3_bootstrap.csv` - Per-age-group bootstrap CIs
- `ecsc_age_v3_context_counts.csv` - N docs per context length
- `ecsc_age_v3_curve_cis.csv` - Bootstrap CIs for perplexity curves
- `ecsc_age_v3_longitudinal.csv` - Within-child paired changes (if available)
- `ecsc_age_v3_config.json` - Analysis configuration
- `ecsc_age_analysis_v3_robust.png` - Visualization

## Key Parameters
- `MAX_CONTEXT_FIXED = 512` - primary fixed max for cross-doc comparison
- `MAX_CONTEXT_LONG = 1024` - secondary fixed max for longer documents
- `BENEFIT_EPS = 1.0` - minimum perplexity benefit to compute valid half-life
- `MIN_DOCS_FOR_PLOT = 10` - threshold for solid vs faded plotting